In [ ]:
!pip install s3fs pyarrow xgboost scikit-learn matplotlib seaborn -q

import pandas as pd
import numpy as np
import s3fs
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import gc
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Connecting to S3 bucket
fs = s3fs.S3FileSystem(
    anon=False,
    key='your_key',
    secret='your_secret_key',
    client_kwargs={'region_name': 'us-west-1'}
)

print("S3 connected!")

In [ ]:
COLS_NEEDED = [
    'date', 'store_id', 'item_id', 'unit_sales',
    'onpromotion', 'family', 'perishable',
    'oil_price', 'transactions',
    'holiday_type', 'locale',
    'city_norm',
    'temp_avg_c', 'temp_min_c', 'temp_max_c',
    'humidity_avg_pct', 'precip_total_mm',
]

df = pq.ParquetDataset(
    'team-massachusetts-favorita/model_base.parquet/',
    filesystem=fs
).read(columns=COLS_NEEDED).to_pandas()

print(f"Loaded: {df.shape}")
print(f"Memory: {df.memory_usage(deep=True).sum()/1e9:.2f} GB")

In [ ]:
df['unit_sales']       = df['unit_sales'].astype('float32')
df['oil_price']        = df['oil_price'].astype('float32')
df['temp_avg_c']       = df['temp_avg_c'].astype('float32')
df['temp_min_c']       = df['temp_min_c'].astype('float32')
df['temp_max_c']       = df['temp_max_c'].astype('float32')
df['humidity_avg_pct'] = df['humidity_avg_pct'].astype('float32')
df['precip_total_mm']  = df['precip_total_mm'].astype('float32')
df['transactions']     = df['transactions'].astype('float32')
df['store_id']         = df['store_id'].astype('int32')
df['item_id']          = df['item_id'].astype('int32')

gc.collect()
print(f"Dtypes downcasted!")
print(f"Memory: {df.memory_usage(deep=True).sum()/1e9:.2f} GB")

In [ ]:
# Convert date
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['store_id', 'item_id', 'date']).reset_index(drop=True)

# Fill values 
df['oil_price'] = df['oil_price'].fillna(method='ffill').fillna(df['oil_price'].median())

for col in ['temp_avg_c', 'temp_min_c', 'temp_max_c', 'humidity_avg_pct', 'precip_total_mm']:
    df[col] = df.groupby('city_norm')[col].transform(lambda x: x.fillna(x.median()))

df['transactions'] = df.groupby('store_id')['transactions'].transform(
    lambda x: x.fillna(x.median())
)

# Convert booleans 
df['onpromotion'] = df['onpromotion'].fillna(False).astype('int8')
df['perishable']  = df['perishable'].fillna(False).astype('int8')

# Holiday flags 
df['is_holiday']          = (df['holiday_type'].notna()).astype('int8')
df['is_national_holiday'] = (df['locale'] == 'National').astype('int8')
df['is_regional_holiday'] = (df['locale'] == 'Regional').astype('int8')
df['is_local_holiday']    = (df['locale'] == 'Local').astype('int8')

# Drop unwanted columns 
df.drop(columns=['holiday_type', 'locale'], inplace=True)

gc.collect()
print("Cleaning done!")
print(f"Memory: {df.memory_usage(deep=True).sum()/1e9:.2f} GB")

In [ ]:
df['day_of_week']    = df['date'].dt.dayofweek.astype('int8')
df['month']          = df['date'].dt.month.astype('int8')
df['year']           = df['date'].dt.year.astype('int16')
df['day_of_month']   = df['date'].dt.day.astype('int8')
df['week_of_year']   = df['date'].dt.isocalendar().week.astype('int8')
df['is_weekend']     = (df['day_of_week'] >= 5).astype('int8')
df['is_month_start'] = df['date'].dt.is_month_start.astype('int8')
df['is_month_end']   = df['date'].dt.is_month_end.astype('int8')
df['quarter']        = df['date'].dt.quarter.astype('int8')

gc.collect()
print("Time features done!")

In [ ]:
grp = df.groupby(['store_id', 'item_id'])['unit_sales']

# Lag features
df['lag_7']   = grp.shift(7).astype('float32');  gc.collect()
df['lag_14']  = grp.shift(14).astype('float32'); gc.collect()
df['lag_28']  = grp.shift(28).astype('float32'); gc.collect()
df['lag_365'] = grp.shift(365).astype('float32');gc.collect()

# Rolling means
df['rolling_mean_7']  = grp.shift(1).transform(lambda x: x.rolling(7,  min_periods=1).mean()).astype('float32'); gc.collect()
df['rolling_mean_14'] = grp.shift(1).transform(lambda x: x.rolling(14, min_periods=1).mean()).astype('float32'); gc.collect()
df['rolling_mean_28'] = grp.shift(1).transform(lambda x: x.rolling(28, min_periods=1).mean()).astype('float32'); gc.collect()

# Rolling std 
df['rolling_std_7']  = grp.shift(1).transform(lambda x: x.rolling(7, min_periods=2).std()).astype('float32');  gc.collect()
df['rolling_max_7']  = grp.shift(1).transform(lambda x: x.rolling(7, min_periods=1).max()).astype('float32');  gc.collect()
df['rolling_min_7']  = grp.shift(1).transform(lambda x: x.rolling(7, min_periods=1).min()).astype('float32');  gc.collect()

print(f"Lag & rolling done!")
print(f"Memory: {df.memory_usage(deep=True).sum()/1e9:.2f} GB")

In [ ]:
print("Computing promotion time features...")

# Days since last promotion 
df['promo_date'] = df['onpromotion'] * df['date'].astype('int64')
df['promo_date'] = df['promo_date'].replace(0, pd.NaT.value)

df['days_since_promo'] = (
    df.groupby(['store_id', 'item_id'])['promo_date']
    .transform(lambda x: x.where(x != pd.NaT.value).ffill())
)
df['days_since_promo'] = (
    (df['date'].astype('int64') - df['days_since_promo']) / 1e9 / 86400
).clip(0, 365).fillna(365).astype('float32')
df.drop(columns=['promo_date'], inplace=True)

# Days to next promotion per store+item
df['next_promo_date'] = df['onpromotion'] * df['date'].astype('int64')
df['next_promo_date'] = df['next_promo_date'].replace(0, pd.NaT.value)

df['days_to_promo'] = (
    df.groupby(['store_id', 'item_id'])['next_promo_date']
    .transform(lambda x: x.where(x != pd.NaT.value).bfill())
)
df['days_to_promo'] = (
    (df['days_to_promo'] - df['date'].astype('int64')) / 1e9 / 86400
).clip(0, 365).fillna(365).astype('float32')
df.drop(columns=['next_promo_date'], inplace=True)

gc.collect()
print("Promotion time features done!")

In [ ]:
print("Computing group average features (leakage-free)...")

SPLIT_DATE = '2017-01-01'
train_only = df[df['date'] < SPLIT_DATE]
mask = (train_only['is_holiday'] == 0) & (train_only['onpromotion'] == 0)

#all averages from training data 
base  = train_only[mask].groupby(['store_id','day_of_week'])['unit_sales'].mean().rename('avg_sales_store_dow')
base2 = train_only[mask].groupby(['item_id','day_of_week'])['unit_sales'].mean().rename('avg_sales_item_dow')
base3 = train_only[mask].groupby(['store_id','month'])['unit_sales'].mean().rename('avg_sales_store_month')

promo_avg    = train_only[train_only['onpromotion']==1].groupby('item_id')['unit_sales'].mean()
no_promo_avg = train_only[train_only['onpromotion']==0].groupby('item_id')['unit_sales'].mean()
promo_factor = (promo_avg - no_promo_avg).rename('promo_factor')

# Merging into full df
df = df.merge(base.reset_index(),         on=['store_id','day_of_week'], how='left')
df = df.merge(base2.reset_index(),        on=['item_id','day_of_week'],  how='left')
df = df.merge(base3.reset_index(),        on=['store_id','month'],       how='left')
df = df.merge(promo_factor.reset_index(), on='item_id',                  how='left')

df['promo_factor']          = df['promo_factor'].fillna(0).astype('float32')
df['avg_sales_store_dow']   = df['avg_sales_store_dow'].astype('float32')
df['avg_sales_item_dow']    = df['avg_sales_item_dow'].fillna(df['avg_sales_item_dow'].median()).astype('float32')
df['avg_sales_store_month'] = df['avg_sales_store_month'].astype('float32')

gc.collect()
print("Group average features done (train-only, no leakage)!")
print(f"Memory: {df.memory_usage(deep=True).sum()/1e9:.2f} GB")

In [ ]:
df['lag7_x_promo']    = (df['lag_7']          * df['onpromotion']).astype('float32')
df['lag7_x_oil']      = (df['lag_7']          * df['oil_price']).astype('float32')
df['lag7_x_humid']    = (df['lag_7']          * df['humidity_avg_pct']).astype('float32')
df['rolling_x_promo'] = (df['rolling_mean_7'] * df['onpromotion']).astype('float32')
df['rolling_x_temp']  = (df['rolling_mean_7'] * df['temp_avg_c']).astype('float32')
df['temp_range']      = (df['temp_max_c']     - df['temp_min_c']).astype('float32')

# Spike flag
df['rolling_std_7']   = df['rolling_std_7'].fillna(0)
df['is_spike']        = (
    df['unit_sales'] > df['rolling_mean_7'] + 2 * df['rolling_std_7']
).astype('int8')

gc.collect()
print("Interaction features done!")
print(f"Total features: {len(df.columns)}")
print(f"Memory: {df.memory_usage(deep=True).sum()/1e9:.2f} GB")

In [ ]:
# Loading external datasets
df_imports = pq.read_table('team-massachusetts-favorita/Ecuador_import.parquet', filesystem=fs).to_pandas()
df_exports = pq.read_table('team-massachusetts-favorita/ecuador_exports.parquet', filesystem=fs).to_pandas()
df_production = pq.read_table('team-massachusetts-favorita/ecuador_production.parquet', filesystem=fs).to_pandas()

# Aggregating imports/exports by month + product
df_imports_agg = df_imports.groupby(['refYear','refMonth','cmdCode'])['primaryValue'].sum().reset_index()
df_imports_agg.columns = ['year','month','cmdCode','import_value']

df_exports_agg = df_exports.groupby(['refYear','refMonth','cmdCode'])['primaryValue'].sum().reset_index()
df_exports_agg.columns = ['year','month','cmdCode','export_value']

# Map Favorita families to HS codes
family_to_cmd = {
    'PRODUCE': '07', 'FRUITS': '08', 'DAIRY': '04',
    'MEATS': '02', 'BREAD/BAKERY': '07', 'EGGS': '04',
}
df['cmdCode'] = df['family'].map(family_to_cmd).fillna('07')

# Merge
df = df.merge(df_imports_agg, on=['year','month','cmdCode'], how='left')
df = df.merge(df_exports_agg, on=['year','month','cmdCode'], how='left')
df['import_value'] = df['import_value'].fillna(0).astype('float32')
df['export_value'] = df['export_value'].fillna(0).astype('float32')

# Production - annual 
df_prod_pivot = df_production.pivot_table(
    index='Year', columns='Item', values='Value', aggfunc='sum'
).reset_index().rename(columns={'Year': 'year'})
df_prod_pivot.columns.name = None
prod_cols = ['year','Bananas','Tomatoes','Potatoes',
             'Meat of cattle with the bone, fresh or chilled','Raw milk of cattle']
prod_cols = [c for c in prod_cols if c in df_prod_pivot.columns]
df_prod_pivot = df_prod_pivot[prod_cols]
df_prod_pivot.columns = ['year'] + [f'prod_{c[:15].lower().replace(" ","_")}'
                                      for c in df_prod_pivot.columns[1:]]
df = df.merge(df_prod_pivot, on='year', how='left')
for c in df_prod_pivot.columns[1:]:
    df[c] = df[c].astype('float32')

gc.collect()
print("External signals merged!")
print(f"Shape: {df.shape}")

In [ ]:
# Drop rows with missing lag features
df = df.dropna(subset=['lag_7', 'lag_14', 'rolling_mean_7'])

#time-based split — 2013-2016 train, 2017 validate
SPLIT_DATE = '2017-01-01'
train = df[df['date'] <  SPLIT_DATE]
val   = df[df['date'] >= SPLIT_DATE]

print(f"Train: {train.shape} | {train['date'].min()} → {train['date'].max()}")
print(f"Val:   {val.shape}   | {val['date'].min()} → {val['date'].max()}")

In [ ]:
FEATURES = [
    # Lags
    'lag_7', 'lag_14', 'lag_28', 'lag_365',
    # Rolling
    'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28',
    'rolling_std_7', 'rolling_max_7', 'rolling_min_7',
    # Promotion time features (new — from top Kaggle solutions)
    'days_since_promo', 'days_to_promo', 'promo_factor',
    # Group averages (new — from top Kaggle solutions)
    'avg_sales_store_dow', 'avg_sales_item_dow',
    'avg_sales_store_month',
    # Interactions
    'lag7_x_promo', 'lag7_x_oil', 'lag7_x_humid',
    'rolling_x_promo', 'rolling_x_temp',
    # Weather
    'temp_avg_c', 'temp_min_c', 'temp_max_c',
    'humidity_avg_pct', 'precip_total_mm', 'temp_range',
    # Economic
    'oil_price', 'transactions',
    'import_value', 'export_value',
    # Time
    'day_of_week', 'month', 'year', 'day_of_month',
    'week_of_year', 'is_weekend', 'is_month_start',
    'is_month_end', 'quarter',
    # Product/store
    'onpromotion', 'perishable', 'store_id', 'item_id',
    # Holiday (3 levels)
    'is_holiday', 'is_national_holiday',
    'is_regional_holiday', 'is_local_holiday',
    'is_spike',
]

# Add production columns
prod_feature_cols = [c for c in df.columns if c.startswith('prod_')]
FEATURES += prod_feature_cols

# Keeping only existing columns
FEATURES = [f for f in FEATURES if f in df.columns]

X_train = train[FEATURES]
y_train = train['unit_sales']
X_val   = val[FEATURES]
y_val   = val['unit_sales']

print(f"Training with {len(FEATURES)} features")
print(f"X_train: {X_train.shape} | X_val: {X_val.shape}")

In [ ]:
# Transform target
y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)

model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=30,
    eval_metric='mae',   # keep this
    tree_method='hist',
    device='cuda'
)

# Train using LOG target
model.fit(
    X_train, y_train_log,
    eval_set=[(X_val, y_val_log)],
    verbose=100
)

print("Training done!")

In [ ]:
# Predict in log space
preds_log = model.predict(X_val)

# Convert back to original scale
preds = np.expm1(preds_log)

mae   = mean_absolute_error(y_val, preds)
rmse  = np.sqrt(mean_squared_error(y_val, preds))
wmape = np.sum(np.abs(y_val - preds)) / np.sum(y_val) * 100

print("=" * 45)
print(f"  MAE:       {mae:.4f}")
print(f"  RMSE:      {rmse:.4f}")
print(f"  WMAPE:     {wmape:.2f}%")
print("=" * 45)

print(f"\n  Phase II MAE:   4.0893")
print(f"  Phase III MAE:  {mae:.4f}")
print(f"  Improvement:    {((4.0893 - mae) / 4.0893 * 100):.1f}%")

In [ ]:
importance_df = pd.DataFrame({
    'feature': FEATURES,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False).head(20)

plt.figure(figsize=(10, 8))
sns.barplot(data=importance_df, x='importance', y='feature', palette='viridis')
plt.title('Top 20 Feature Importances — Phase III XGBoost')
plt.xlabel('Relative Importance')
plt.tight_layout()
plt.show()

In [ ]:
import shap

explainer = shap.Explainer(model)
shap_values = explainer(X_val)

shap.summary_plot(shap_values, X_val)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

#model metrics
mae = mean_absolute_error(y_val, preds)
rmse = np.sqrt(mean_squared_error(y_val, preds))
wmape = np.sum(np.abs(y_val - preds)) / np.sum(y_val) * 100

# safer MAPE / SMAPE handling
eps = 1e-8

mape = np.mean(np.abs((y_val - preds) / (y_val + eps))) * 100

smape = 100 * np.mean(
    2 * np.abs(preds - y_val) / (np.abs(y_val) + np.abs(preds) + eps)
)

r2 = r2_score(y_val, preds)
bias = np.mean(preds - y_val)


# Baseline metrics (lag_7)
baseline_preds = X_val['lag_7']

baseline_mae = mean_absolute_error(y_val, baseline_preds)
baseline_rmse = np.sqrt(mean_squared_error(y_val, baseline_preds))
baseline_wmape = np.sum(np.abs(y_val - baseline_preds)) / np.sum(y_val) * 100

baseline_mape = np.mean(np.abs((y_val - baseline_preds) / (y_val + eps))) * 100

baseline_smape = 100 * np.mean(
    2 * np.abs(baseline_preds - y_val) / (np.abs(y_val) + np.abs(baseline_preds) + eps)
)

baseline_r2 = r2_score(y_val, baseline_preds)
baseline_bias = np.mean(baseline_preds - y_val)

# Comparison table
results_table = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'WMAPE (%)', 'MAPE (%)', 'SMAPE (%)', 'R²'],
    'Baseline (lag_7)': [
        baseline_mae,
        baseline_rmse,
        baseline_wmape,
        baseline_mape,
        baseline_smape,
        baseline_r2
    ],
    'XGBoost Model': [
        mae,
        rmse,
        wmape,
        mape,
        smape,
        r2
    ]
})

results_table['Improvement (%)'] = [
    ((baseline_mae - mae) / baseline_mae) * 100,
    ((baseline_rmse - rmse) / baseline_rmse) * 100,
    ((baseline_wmape - wmape) / baseline_wmape) * 100,
    ((baseline_mape - mape) / baseline_mape) * 100,
    ((baseline_smape - smape) / baseline_smape) * 100,
    ((r2 - baseline_r2) / (abs(baseline_r2) + eps)) * 100
]

results_table.round(4)

In [ ]:
print(X_val.columns)

In [ ]:
import joblib
import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np

joblib.dump(model, 'xgboost_phase3.pkl')
fs.put('xgboost_phase3.pkl', 'team-massachusetts-favorita/models/xgboost_phase3.pkl')


# Saving metadata
model_info = {
    "model": "XGBoost Phase III",
    "features": list(X_train.columns),
    "metrics": {
        "MAE": mae,
        "RMSE": rmse,
        "WMAPE": wmape
    }
}

joblib.dump(model_info, 'model_metadata.pkl')
fs.put('model_metadata.pkl', 'team-massachusetts-favorita/models/model_metadata.pkl')

# Saving predictions
val_preds = val.copy()

val_preds['actual'] = y_val
val_preds['predicted'] = preds
val_preds['error'] = val_preds['actual'] - val_preds['predicted']
val_preds['abs_error'] = np.abs(val_preds['error'])

pq.write_table(
    pa.Table.from_pandas(val_preds),
    'team-massachusetts-favorita/outputs/forecast_output.parquet',
    filesystem=fs
)

print("Model saved!")
print("Metadata saved!")
print("Forecast output saved!")
print(f"   Shape: {val_preds.shape}")

In [ ]:
df_results = val.copy()

df_results['actual'] = y_val
df_results['predicted'] = preds

In [ ]:
import numpy as np

np.random.seed(42) 

df_results['supply'] = df_results['predicted'] * np.random.uniform(0.9, 1.1, len(df_results))
df_results['gap'] = df_results['predicted'] - df_results['supply']

df_results['waste'] = df_results['gap'].apply(lambda x: abs(x) if x < 0 else 0)
df_results['stockout'] = df_results['gap'].apply(lambda x: x if x > 0 else 0)

In [ ]:
def recommend(row):
    if row['gap'] > 0:
        return "Increase supply"
    elif row['gap'] < 0:
        return "Reduce supply"
    else:
        return "Balanced"

df_results['recommendation'] = df_results.apply(recommend, axis=1)

In [ ]:
df_daily = df_results.groupby('date').agg({
    'predicted': 'sum',
    'actual': 'sum',
    'waste': 'sum',
    'stockout': 'sum'
}).reset_index()

In [ ]:
print("Total Waste:", df_results['waste'].sum())
print("Total Stockout:", df_results['stockout'].sum())
print("Avg Waste per Record:", df_results['waste'].mean())
print("Avg Stockout per Record:", df_results['stockout'].mean())

print("Waste Cases:", (df_results['waste'] > 0).sum())
print("Stockout Cases:", (df_results['stockout'] > 0).sum())
print("\nRecommendation Distribution:")
print(df_results['recommendation'].value_counts())

In [ ]:
total = len(df_results)

print("\nPercentage Metrics:")
print("Waste %:", (df_results['waste'] > 0).sum() / total * 100)
print("Stockout %:", (df_results['stockout'] > 0).sum() / total * 100)

In [ ]:
def calculate_location_metrics(df, group_col):
    metrics = df.groupby(group_col).agg(
        Total_Records=(group_col, 'size'),
        Waste_Cases=('waste', lambda x: (x > 0).sum()),
        Stockout_Cases=('stockout', lambda x: (x > 0).sum())
    )
    metrics['Waste %'] = (metrics['Waste_Cases'] / metrics['Total_Records']) * 100
    metrics['Stockout %'] = (metrics['Stockout_Cases'] / metrics['Total_Records']) * 100
    return metrics.sort_values('Waste %', ascending=False)

# Calculate for each City (using city_norm)
city_analysis = calculate_location_metrics(df_results, 'city_norm')
print("--- City-Level Metrics ---")
print(city_analysis[['Total_Records', 'Waste %', 'Stockout %']])

In [ ]:
import numpy as np
import pandas as pd


# INVENTORY SIMULATION

df_results = df_results.copy()

# Gap convention:
#   gap > 0  = actual > predicted -> understock -> increase supply
#   gap < 0  = predicted > actual -> overstock -> reduce supply
df_results["gap"] = df_results["actual"] - df_results["predicted"]

# Waste = overprediction
df_results["waste"] = np.maximum(df_results["predicted"] - df_results["actual"], 0)

# Stockout = underprediction
df_results["stockout"] = np.maximum(df_results["actual"] - df_results["predicted"], 0)

# Recommendation rule
def recommend(row):
    if row["gap"] > 0:
        return "Increase supply"
    elif row["gap"] < 0:
        return "Reduce supply"
    else:
        return "Balanced"

df_results["recommendation"] = df_results.apply(recommend, axis=1)

#daily aggregation if date exists
if "date" in df_results.columns:
    df_daily = df_results.groupby("date", as_index=False).agg({
        "predicted": "sum",
        "actual": "sum",
        "waste": "sum",
        "stockout": "sum"
    })
else:
    df_daily = None

# Summary metrics
total_records = len(df_results)
total_actual = df_results["actual"].sum()

summary = {
    "total_waste": df_results["waste"].sum(),
    "total_stockout": df_results["stockout"].sum(),
    "avg_waste_per_record": df_results["waste"].mean(),
    "avg_stockout_per_record": df_results["stockout"].mean(),
    "waste_cases": (df_results["waste"] > 0).sum(),
    "stockout_cases": (df_results["stockout"] > 0).sum(),
    "waste_rate_pct": (df_results["waste"].sum() / total_actual) * 100 if total_actual else 0,
    "stockout_rate_pct": (df_results["stockout"].sum() / total_actual) * 100 if total_actual else 0
}

print("=== Simulation Summary ===")
for k, v in summary.items():
    print(f"{k}: {v}")

print(df_results["recommendation"].value_counts())

display(df_results[["actual", "predicted", "gap", "waste", "stockout", "recommendation"]].head())

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Actual values
actual = val["unit_sales"].values

# Predictions
lag_preds = val["lag_7"].values
rolling_preds = val["rolling_mean_7"].values
xgb_preds = preds

def compute_metrics(name, actual, predicted):
    actual = np.asarray(actual)
    predicted = np.asarray(predicted)

    mae = mean_absolute_error(actual, predicted)

    waste = np.maximum(predicted - actual, 0)
    stockout = np.maximum(actual - predicted, 0)

    total_actual = actual.sum()

    return {
        "Model": name,
        "MAE": round(mae, 3),
        "Waste %": round((waste.sum() / total_actual) * 100, 2),
        "Stockout %": round((stockout.sum() / total_actual) * 100, 2)
    }

# Comparison table
comparison_table = pd.DataFrame([
    compute_metrics("Lag-7", actual, lag_preds),
    compute_metrics("Rolling Mean", actual, rolling_preds),
    compute_metrics("XGBoost", actual, xgb_preds)
])

print(comparison_table)

In [ ]:
print(df.columns.tolist())
print(df.dtypes)
print(df.head(2))

In [ ]:
import joblib
import os

os.makedirs('data/model_artifacts', exist_ok=True)

# Save model
joblib.dump(model, 'data/model_artifacts/demand_model.pkl')
joblib.dump(FEATURES, 'data/model_artifacts/feature_list.pkl')

# Save feature lookup table 
latest_features = (
    df.sort_values('date')
    .groupby(['store_id', 'item_id'])
    .last()
    .reset_index()
)
latest_features.to_parquet('data/sample_features.parquet', index=False)

print(f"Feature list saved: {len(FEATURES)} features")
print(f"Feature lookup saved: {latest_features.shape[0]} rows")